# Notebook 1: NVIDIA NIM Inference — Financial QA Baseline
---

**Objective**: Connect to the NVIDIA NIM microservice and query it with financial questions from the FinanceBench dataset. This establishes our baseline model responses before any optimization.

**DLI Course Mapping**: Learning Objective 1 — "Query a model using NVIDIA NIM microservice"

**What you'll do**:
1. Set up the NIM inference client
2. Load the FinanceBench dataset from Hugging Face
3. Generate baseline responses for all test questions
4. Save responses for evaluation in Notebook 2

In [ ]:
# ============================================================
# Setup & Imports — From NVIDIA DLI Course Lab
# ============================================================
import os
import sys
import json
import time
import pandas as pd
from pathlib import Path

# Add project root to path
PROJECT_ROOT = Path("..").resolve()
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from utils import (
    NIMInferenceClient,
    load_financebench,
    format_finance_prompt,
    set_seed,
    save_results,
    print_metrics_summary,
    RESULTS_DIR,
)

# Reproducibility
set_seed(42)

print(f"Project root: {PROJECT_ROOT}")
print(f"Results dir:  {RESULTS_DIR}")

## 1. Configure NVIDIA NIM API Access

You need an NVIDIA API key from [build.nvidia.com](https://build.nvidia.com). The free developer tier provides sufficient credits for this project.

> **From DLI Course**: The NIM microservice exposes an OpenAI-compatible API endpoint, making it a drop-in replacement for any OpenAI-based workflow.

In [ ]:
# ============================================================
# NIM Client Setup — From NVIDIA DLI Course Lab
# ============================================================

# Set your API key (get it from https://build.nvidia.com)
# Option 1: Environment variable (recommended)
# export NVIDIA_API_KEY="nvapi-YOUR_KEY_HERE"

# Option 2: Direct assignment (for quick testing only)
# os.environ["NVIDIA_API_KEY"] = "nvapi-YOUR_KEY_HERE"

# Initialize the NIM client
# Using Llama-3.1-8B-Instruct as our base model (same as DLI course labs)
nim_client = NIMInferenceClient(
    model="meta/llama-3.1-8b-instruct",
)

# Quick connectivity test
test_response = nim_client.query("What is EBITDA? Answer in one sentence.")
print(f"NIM Test Response: {test_response}")

## 2. Load FinanceBench Dataset

[PatronusAI/financebench](https://huggingface.co/datasets/PatronusAI/financebench) contains 150+ real financial QA examples from SEC 10-K and 10-Q filings with ground-truth answers and evidence passages.

In [ ]:
# ============================================================
# Load & Split Dataset
# ============================================================

train_df, test_df = load_financebench(split_ratio=0.8, seed=42)

print(f"\nTrain set: {len(train_df)} examples")
print(f"Test set:  {len(test_df)} examples")
print(f"\nColumns: {list(test_df.columns)}")
print(f"\nSample question: {test_df.iloc[0]['question'][:200]}")
print(f"Sample answer:   {test_df.iloc[0]['answer'][:200]}")

In [ ]:
# ============================================================
# Explore the dataset
# ============================================================

print("=== Dataset Statistics ===")
print(f"Total examples: {len(train_df) + len(test_df)}")
print(f"\nQuestion length (chars):")
print(test_df["question"].str.len().describe().round(1))
print(f"\nAnswer length (chars):")
print(test_df["answer"].str.len().describe().round(1))

# Show a few examples
print("\n=== Sample QA Pairs ===")
for i in range(min(3, len(test_df))):
    row = test_df.iloc[i]
    print(f"\n--- Example {i+1} ---")
    print(f"Q: {row['question'][:300]}")
    print(f"A: {row['answer'][:300]}")
    if row.get('evidence'):
        print(f"E: {str(row['evidence'])[:200]}...")

## 3. Generate Baseline Responses via NIM

We query the NIM endpoint with each test question formatted with available context/evidence. The system prompt instructs the model to act as a financial analyst.

> **From DLI Course**: Using a domain-specific system prompt is the simplest form of prompt engineering before ICL or fine-tuning.

In [ ]:
# ============================================================
# Format prompts for baseline inference (no ICL examples)
# ============================================================

test_prompts = []
for _, row in test_df.iterrows():
    prompt = format_finance_prompt(
        question=row["question"],
        context=row.get("context", ""),
        evidence=row.get("evidence", ""),
        icl_examples=None,  # No ICL for baseline
    )
    test_prompts.append(prompt)

print(f"Formatted {len(test_prompts)} prompts")
print(f"\n--- Sample Prompt ---\n{test_prompts[0][:500]}...")

In [ ]:
# ============================================================
# Batch Query NIM — Baseline Inference
# ============================================================

print("Starting baseline inference via NVIDIA NIM...")
print(f"Model: {nim_client.model}")
print(f"Questions: {len(test_prompts)}")

start_time = time.time()

baseline_responses = nim_client.batch_query(
    prompts=test_prompts,
    system_prompt=(
        "You are a precise financial analyst specializing in SEC filings. "
        "Answer questions accurately and concisely based on the provided context. "
        "If numerical data is involved, be exact with figures."
    ),
    temperature=0.1,
    max_tokens=512,
    delay=0.5,  # Rate limiting
)

elapsed = time.time() - start_time
print(f"\nBaseline inference complete in {elapsed:.1f}s")
print(f"Average latency: {elapsed/len(test_prompts):.2f}s per query")

In [ ]:
# ============================================================
# Review Baseline Responses
# ============================================================

print("=== Baseline Response Samples ===\n")
for i in range(min(5, len(test_df))):
    print(f"--- Example {i+1} ---")
    print(f"Q: {test_df.iloc[i]['question'][:200]}")
    print(f"Expected: {test_df.iloc[i]['answer'][:200]}")
    print(f"Got:      {baseline_responses[i][:200]}")
    print()

# Check for errors
errors = [r for r in baseline_responses if r.startswith("[ERROR]")]
print(f"Errors: {len(errors)} / {len(baseline_responses)}")

## 4. Save Results

Save all baseline responses for evaluation in Notebook 2.

In [ ]:
# ============================================================
# Save Baseline Results
# ============================================================

# Create results DataFrame
baseline_results = test_df.copy()
baseline_results["model_response"] = baseline_responses
baseline_results["model_config"] = "base_llama31_8b"
baseline_results["timestamp"] = pd.Timestamp.now().isoformat()

# Save to results directory
save_results(baseline_results, "baseline_responses.csv")

# Also save the train/test split for consistency across notebooks
save_results(train_df, "train_split.csv")
save_results(test_df, "test_split.csv")

print(f"\nSaved {len(baseline_results)} baseline responses to results/baseline_responses.csv")
print("Saved train/test splits for reproducibility.")
print("\n✓ Notebook 1 complete — proceed to Notebook 2 for evaluation.")